# Spikes Dataset: SSIT Model Analysis

This notebook completes the noise decomposition analysis for the spikes dataset.

**Prerequisites**: Run `spikes_descriptive_analysis.py` first to prepare the data.

## Overview

- **Dataset**: 1000 spike identification questions, 37 participants (13 Experts, 17 Experienced, 7 Novices)
- **Model**: SSIT (appropriate for T=1, single-occasion data)
- **Estimates**:
  - Level noise (bias) for each participant
  - Population skill level (1/σ_residual)
  - Item difficulties (signal strength)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from noise_decomposition_flexible import run_analysis
from noise_analysis_lib import (
    extract_participant_estimates,
    extract_item_difficulties,
    plot_variance_decomposition
)

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

## 1. Load Prepared Data

In [2]:
# Load the prepared data
df = pd.read_csv("spikes_analysis_output/spikes_data_for_analysis.csv")

# Load descriptive stats for reference
roc_df = pd.read_csv("spikes_analysis_output/roc_points.csv")

print(f"Data shape: {df.shape}")
print(f"\nFirst few rows:")
df.head(10)

Data shape: (37000, 4)

First few rows:


,id,item,t,Y
0,1,1,1,0
1,1,2,1,0
2,1,3,1,1
3,1,4,1,1
4,1,5,1,1
5,1,6,1,1
6,1,7,1,1
7,1,8,1,0
8,1,9,1,0
9,1,10,1,1


## 2. Run SSIT Model Analysis

This will take 10-20 minutes depending on your hardware.

**Model**: SSIT (Similar Skill Individual Threshold)
- Estimates: Signal (c_i), Level Noise (ℓ_j), Residual (pattern + occasion)
- Appropriate for T=1 (single occasion) data

In [ ]:
# Run the SSIT model
results = run_analysis(
    "spikes_analysis_output/spikes_data_for_analysis.csv",
    model_type="SSIT",
    results_filename="spikes_analysis_output/spikes_SSIT_results.pkl",
    draws=5000,
    tune=2000,
    chains=10,
    cores=6,
    target_accept=0.98,
    random_seed=42
)

In [ ]:
import pickle
import os

results_file = "spikes_analysis_output/spikes_SSIT_results.pkl"

if os.path.exists(results_file):
    print(f"Loading saved results from: {results_file}")
    with open(results_file, "rb") as f:
        results = pickle.load(f)
    print("✓ Results loaded successfully!")
    print(f"  Model type: {results['model_type']}")
    print(f"  Total variance: {results['variance_decomposition']['total_variance']:.3f}")
else:
    print(f"ERROR: Results file not found: {results_file}")
    print("Please run the analysis cell above first.")

## 2b. OR Load Previously Saved Results

If you've already run the analysis and want to skip the long computation, run this cell instead of the one above.

## 3. Variance Decomposition

In [ ]:
# Display variance decomposition
vd = results['variance_decomposition']
scales = results['effective_scales']

print("="*70)
print("VARIANCE DECOMPOSITION (SSIT Model)")
print("="*70)
print(f"\nTotal Variance: {vd['total_variance']:.3f}\n")
print(f"Signal (item difficulty):  {vd['perc_c']:6.2f}%   (σ_c = {scales['S_est']:.3f})")
print(f"Level Noise (bias):        {vd['perc_l']:6.2f}%   (σ_l = {scales['LN_est']:.3f})")
print(f"Residual (other noise):    {vd['perc_p']:6.2f}%   (σ_r = {scales['PN_est']:.3f})")
print(f"\nPopulation Skill (1/σ_r):  {1/scales['PN_est']:.3f}")

In [ ]:
# Plot variance decomposition
fig, ax = plot_variance_decomposition(
    results,
    title="Spikes Dataset: Variance Decomposition (SSIT Model)"
)
plt.show()

## 4. Participant-Level Estimates

In [ ]:
# Load expertise data from original source
data_dir = "/Users/bwestove/cdac Dropbox/brandon westover/0_People/temp_people/fabio/KILT_technical/spikes"
expertise_df = pd.read_csv(f"{data_dir}/expertise_spikes.csv", header=None)
expertise = expertise_df.values.flatten()

# Create expertise mapping (rater ID -> expertise level)
expertise_map = {}
for i in range(len(expertise)):
    expertise_map[i+1] = expertise[i]  # IDs are 1-indexed

# Add expertise to df for extraction
df['expertise'] = df['id'].map(expertise_map)

# Extract participant estimates
participant_df = extract_participant_estimates(results, df)

print("Participant Estimates (first 15):")
print(participant_df.head(15).to_string(index=False))

In [ ]:
# Summary by expertise
expertise_labels = {0: 'Experts', 1: 'Experienced', 2: 'Novices'}
participant_df['expertise_label'] = participant_df['expertise'].map(expertise_labels)

print("\nLevel Noise by Expertise Group:")
ln_summary = participant_df.groupby('expertise_label')['level_noise'].agg([
    'count', 'mean', 'std', 'min', 'max'
])
print(ln_summary)

## 5. Level Noise Visualization

In [ ]:
# Boxplot of level noise by expertise
fig, ax = plt.subplots(figsize=(10, 6))

sns.boxplot(data=participant_df, x='expertise_label', y='level_noise',
           order=['Experts', 'Experienced', 'Novices'], ax=ax)
sns.swarmplot(data=participant_df, x='expertise_label', y='level_noise',
             order=['Experts', 'Experienced', 'Novices'],
             color='black', alpha=0.5, ax=ax)

ax.set_xlabel('Expertise Level', fontsize=14)
ax.set_ylabel('Level Noise (Bias)', fontsize=14)
ax.set_title('Level Noise Distribution by Expertise', fontsize=16, fontweight='bold')
ax.axhline(0, color='red', linestyle='--', alpha=0.5, label='No bias (neutral threshold)')
ax.legend(fontsize=12)
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('spikes_analysis_output/level_noise_by_expertise_SSIT.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Item Difficulty Estimates

In [ ]:
# Extract item difficulties
item_df = extract_item_difficulties(results)

print(f"Extracted difficulties for {len(item_df)} items\n")
print("Difficulty distribution:")
print(item_df['difficulty'].describe())

print("\nEasiest items (most negative difficulty = most likely 'spike'):")
print(item_df.nsmallest(10, 'difficulty')[['item_id', 'difficulty']].to_string(index=False))

print("\nHardest items (most positive difficulty = least likely 'spike'):")
print(item_df.nlargest(10, 'difficulty')[['item_id', 'difficulty']].to_string(index=False))

In [ ]:
# Histogram of item difficulties
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(item_df['difficulty'], bins=50, alpha=0.7, edgecolor='black')
ax.axvline(0, color='red', linestyle='--', linewidth=2, label='Neutral difficulty')
ax.set_xlabel('Item Difficulty (Signal Strength)', fontsize=14)
ax.set_ylabel('Count', fontsize=14)
ax.set_title('Distribution of Item Difficulties', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('spikes_analysis_output/item_difficulty_distribution_SSIT.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Accuracy vs Level Noise

Examine relationship between estimated bias and observed accuracy.

In [ ]:
# Merge with ROC data
merged_df = participant_df.merge(
    roc_df[['Rater ID', 'Accuracy', 'Sensitivity (TPR)', 'FPR']],
    left_on='rater_id',
    right_on='Rater ID',
    how='left'
)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
colors = {'Experts': '#2E86AB', 'Experienced': '#F18F01', 'Novices': '#C73E1D'}

for expertise_label in ['Experts', 'Experienced', 'Novices']:
    group_data = merged_df[merged_df['expertise_label'] == expertise_label]
    ax.scatter(group_data['level_noise'], group_data['Accuracy'],
               alpha=0.7, s=120, label=expertise_label, color=colors[expertise_label])

ax.set_xlabel('Level Noise (Bias)', fontsize=14)
ax.set_ylabel('Accuracy', fontsize=14)
ax.set_title('Accuracy vs Level Noise by Expertise', fontsize=16, fontweight='bold')
ax.axvline(0, color='red', linestyle='--', alpha=0.3, label='No bias')
ax.legend(fontsize=12)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('spikes_analysis_output/accuracy_vs_level_noise_SSIT.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Save Results

In [ ]:
# Save participant estimates
participant_df.to_csv('spikes_analysis_output/participant_estimates_SSIT.csv', index=False)

# Save item difficulties
item_df.to_csv('spikes_analysis_output/item_difficulties_SSIT.csv', index=False)

# Save comprehensive summary
with pd.ExcelWriter('spikes_analysis_output/spikes_SSIT_analysis_complete.xlsx') as writer:
    # Variance decomposition
    vd_df = pd.DataFrame([{
        'Model': 'SSIT',
        'Total Variance': vd['total_variance'],
        'Signal %': vd['perc_c'],
        'Level Noise %': vd['perc_l'],
        'Residual %': vd['perc_p'],
        'σ_signal': scales['S_est'],
        'σ_level': scales['LN_est'],
        'σ_residual': scales['PN_est'],
        'Population Skill': 1/scales['PN_est']
    }])
    vd_df.to_excel(writer, sheet_name='Variance_Decomposition', index=False)
    
    # Participant estimates
    participant_df.to_excel(writer, sheet_name='Participant_Estimates', index=False)
    
    # Item difficulties
    item_df.to_excel(writer, sheet_name='Item_Difficulties', index=False)
    
    # Level noise summary
    ln_summary.to_excel(writer, sheet_name='Level_Noise_Summary')

print("Results saved to: spikes_analysis_output/spikes_SSIT_analysis_complete.xlsx")

## 9. Summary Report

In [ ]:
print("="*70)
print("SPIKES DATASET - SSIT MODEL ANALYSIS SUMMARY")
print("="*70)

print("\n### VARIANCE DECOMPOSITION ###")
print(f"Signal (item difficulty):  {vd['perc_c']:.1f}%   (σ = {scales['S_est']:.3f})")
print(f"Level Noise (bias):        {vd['perc_l']:.1f}%   (σ = {scales['LN_est']:.3f})")
print(f"Residual (other noise):    {vd['perc_p']:.1f}%   (σ = {scales['PN_est']:.3f})")
print(f"\nPopulation Skill (1/σ_r):  {1/scales['PN_est']:.3f}")

print("\n### LEVEL NOISE BY EXPERTISE ###")
for expertise_label in ['Experts', 'Experienced', 'Novices']:
    group_data = participant_df[participant_df['expertise_label'] == expertise_label]
    mean_ln = group_data['level_noise'].mean()
    std_ln = group_data['level_noise'].std()
    median_ln = group_data['level_noise'].median()
    print(f"{expertise_label:12s}: Mean = {mean_ln:+.3f} ± {std_ln:.3f}, Median = {median_ln:+.3f}")

print("\n### KEY FINDINGS ###")
print(f"1. Signal accounts for {vd['perc_c']:.1f}% of variance (item difficulty)")
print(f"2. Level noise accounts for {vd['perc_l']:.1f}% (systematic bias)")
print(f"3. Residual accounts for {vd['perc_p']:.1f}% (pattern + occasion noise)")
print(f"4. Population skill level: {1/scales['PN_est']:.3f}")

# Statistical test for level noise differences
from scipy.stats import kruskal
experts_ln = participant_df[participant_df['expertise_label'] == 'Experts']['level_noise']
experienced_ln = participant_df[participant_df['expertise_label'] == 'Experienced']['level_noise']
novices_ln = participant_df[participant_df['expertise_label'] == 'Novices']['level_noise']

stat, pval = kruskal(experts_ln, experienced_ln, novices_ln)
print(f"\n5. Kruskal-Wallis test for group differences in level noise: p = {pval:.4f}")

print("\n" + "="*70)
print("ANALYSIS COMPLETE")
print("="*70)

In [ ]:
# Save group-level results
group_summary = pd.DataFrame([
    {
        'Group': group_name,
        'N': group_results[group_name]['data']['id'].nunique() if 'data' in group_results[group_name] else 'N/A',
        'Skill': 1/group_results[group_name]['effective_scales']['PN_est'],
        'σ_residual': group_results[group_name]['effective_scales']['PN_est'],
        'σ_level': group_results[group_name]['effective_scales']['LN_est'],
        'σ_signal': group_results[group_name]['effective_scales']['S_est'],
        'Signal %': group_results[group_name]['variance_decomposition']['perc_c'],
        'Level Noise %': group_results[group_name]['variance_decomposition']['perc_l'],
        'Residual %': group_results[group_name]['variance_decomposition']['perc_p']
    }
    for group_name in ['Experts', 'Experienced', 'Novices']
])

# Save to Excel
with pd.ExcelWriter('spikes_analysis_output/group_level_analysis_SSIT.xlsx') as writer:
    group_summary.to_excel(writer, sheet_name='Group_Summary', index=False)
    skill_df.to_excel(writer, sheet_name='Skill_Comparison', index=False)

print("="*70)
print("Group-level results saved to:")
print("  spikes_analysis_output/group_level_analysis_SSIT.xlsx")
print("="*70)
print("\n" + group_summary.to_string(index=False))

In [ ]:
# Visualization: Skill comparison across groups
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Skill levels
ax = axes[0]
groups_list = ['Experts', 'Experienced', 'Novices']
skills = [1/group_results[g]['effective_scales']['PN_est'] for g in groups_list]
colors_plot = ['#2E86AB', '#F18F01', '#C73E1D']

bars = ax.bar(groups_list, skills, color=colors_plot, alpha=0.7, edgecolor='black')
ax.set_ylabel('Skill (1/σ_residual)', fontsize=14)
ax.set_title('Skill Level by Expertise Group', fontsize=16, fontweight='bold')
ax.grid(alpha=0.3, axis='y')

# Add value labels on bars
for bar, skill in zip(bars, skills):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{skill:.3f}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

# Plot 2: Threshold variability (σ_level)
ax = axes[1]
sigma_levels = [group_results[g]['effective_scales']['LN_est'] for g in groups_list]

bars = ax.bar(groups_list, sigma_levels, color=colors_plot, alpha=0.7, edgecolor='black')
ax.set_ylabel('σ_level (Threshold Variability)', fontsize=14)
ax.set_title('Threshold Variability by Expertise Group', fontsize=16, fontweight='bold')
ax.grid(alpha=0.3, axis='y')

# Add value labels on bars
for bar, sigma_l in zip(bars, sigma_levels):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{sigma_l:.3f}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('spikes_analysis_output/group_comparisons_SSIT.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Statistical Test 2: Level Noise Variance Differences (Bayesian approach)
# Compare posterior distributions of σ_level across groups

print("="*70)
print("STATISTICAL TEST 2: LEVEL NOISE VARIANCE DIFFERENCES")
print("="*70)
print("\nBayesian comparison of σ_level (threshold variability)")
print("Higher σ_level = More variable thresholds (less consistent)\n")

# Extract posterior samples for σ_level from each group
sigma_level_samples = {}
for group_name in ['Experts', 'Experienced', 'Novices']:
    trace = group_results[group_name]['trace']
    sigma_level_samples[group_name] = trace.posterior['sigma_l'].values.flatten()

print("Probability that Group A has MORE variable thresholds than Group B:")
print("(i.e., P(σ_level_A > σ_level_B))\n")

for group_a, group_b in comparisons:
    samples_a = sigma_level_samples[group_a]
    samples_b = sigma_level_samples[group_b]
    
    # Probability that σ_level_A > σ_level_B
    prob_a_more_variable = np.mean(samples_a > samples_b)
    
    print(f"  P({group_a} > {group_b}): {prob_a_more_variable:.4f}")
    
    if prob_a_more_variable > 0.95:
        print(f"    → Strong evidence that {group_a} has MORE variable thresholds than {group_b}")
    elif prob_a_more_variable > 0.90:
        print(f"    → Moderate evidence that {group_a} has MORE variable thresholds than {group_b}")
    elif prob_a_more_variable < 0.10:
        print(f"    → Strong evidence that {group_a} has LESS variable thresholds than {group_b}")
    elif prob_a_more_variable < 0.05:
        print(f"    → Moderate evidence that {group_a} has LESS variable thresholds than {group_b}")
    else:
        print(f"    → Weak evidence of difference")
    print()

In [ ]:
# Statistical Test 1: Skill Differences (Bayesian approach)
# Compare posterior distributions of σ_residual across groups

print("="*70)
print("STATISTICAL TEST 1: SKILL DIFFERENCES")
print("="*70)
print("\nBayesian comparison of σ_residual (skill = 1/σ_residual)")
print("Higher σ_residual = Lower skill\n")

# Extract posterior samples for σ_residual from each group
sigma_residual_samples = {}
for group_name in ['Experts', 'Experienced', 'Novices']:
    trace = group_results[group_name]['trace']
    sigma_residual_samples[group_name] = trace.posterior['sigma_residual'].values.flatten()

# Compute probabilities that one group has higher σ_residual (lower skill) than another
comparisons = [
    ('Novices', 'Experts'),
    ('Novices', 'Experienced'),
    ('Experienced', 'Experts')
]

print("Probability that Group A has LOWER skill than Group B:")
print("(i.e., P(σ_residual_A > σ_residual_B))\n")

for group_a, group_b in comparisons:
    samples_a = sigma_residual_samples[group_a]
    samples_b = sigma_residual_samples[group_b]
    
    # Probability that σ_residual_A > σ_residual_B
    prob_a_worse = np.mean(samples_a > samples_b)
    
    print(f"  P({group_a} < {group_b}): {prob_a_worse:.4f}")
    
    if prob_a_worse > 0.95:
        print(f"    → Strong evidence that {group_a} has LOWER skill than {group_b}")
    elif prob_a_worse > 0.90:
        print(f"    → Moderate evidence that {group_a} has LOWER skill than {group_b}")
    elif prob_a_worse < 0.10:
        print(f"    → Strong evidence that {group_a} has HIGHER skill than {group_b}")
    elif prob_a_worse < 0.05:
        print(f"    → Moderate evidence that {group_a} has HIGHER skill than {group_b}")
    else:
        print(f"    → Weak evidence of difference")
    print()

In [ ]:
# Compare skill levels across groups
print("="*70)
print("GROUP-LEVEL SKILL COMPARISON")
print("="*70)
print("\nSkill = 1 / σ_residual (higher = better discrimination)\n")

skill_comparison = []

for group_name in ['Experts', 'Experienced', 'Novices']:
    scales = group_results[group_name]['effective_scales']
    vd = group_results[group_name]['variance_decomposition']
    
    skill = 1 / scales['PN_est']
    sigma_residual = scales['PN_est']
    sigma_level = scales['LN_est']
    
    skill_comparison.append({
        'Group': group_name,
        'Skill': skill,
        'σ_residual': sigma_residual,
        'σ_level': sigma_level,
        'Signal %': vd['perc_c'],
        'Level Noise %': vd['perc_l'],
        'Residual %': vd['perc_p']
    })
    
    print(f"{group_name:12s}: Skill = {skill:.3f}  (σ_residual = {sigma_residual:.3f}, σ_level = {sigma_level:.3f})")

skill_df = pd.DataFrame(skill_comparison)
print("\n" + skill_df.to_string(index=False))

In [ ]:
# Run SSIT analysis separately for each expertise group
import pickle

groups = {
    'Experts': 0,
    'Experienced': 1, 
    'Novices': 2
}

group_results = {}
group_counts = {}

for group_name, expertise_level in groups.items():
    # Check if results already exist
    results_file = f"spikes_analysis_output/spikes_SSIT_{group_name}.pkl"
    
    if os.path.exists(results_file):
        print(f"\n{'='*70}")
        print(f"Loading existing results for {group_name}...")
        print(f"{'='*70}")
        with open(results_file, "rb") as f:
            group_results[group_name] = pickle.load(f)
        print(f"✓ Loaded from {results_file}")
    else:
        # Filter data for this group
        df_group = df[df['expertise'] == expertise_level].copy()
        group_counts[group_name] = df_group['id'].nunique()
        
        # Save to temporary file
        group_file = f"spikes_analysis_output/spikes_data_{group_name}.csv"
        df_group.to_csv(group_file, index=False)
        
        # Run analysis
        print(f"\n{'='*70}")
        print(f"Analyzing {group_name} (n={group_counts[group_name]})...")
        print(f"{'='*70}")
        
        group_results[group_name] = run_analysis(
            group_file,
            model_type="SSIT",
            results_filename=results_file,
            draws=3000,
            tune=1500,
            chains=4,
            cores=4,
            target_accept=0.95,
            random_seed=42
        )

print("\n" + "="*70)
print("GROUP-LEVEL ANALYSIS COMPLETE")
print("="*70)

## 10. Group-Level Analysis by Expertise

Run separate SSIT models for each expertise group to estimate group-specific skill levels and level noise parameters.